<a href="https://colab.research.google.com/github/CarolineMarkov/sepsis-onset-prediction/blob/main/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup - authentication and bigquery connection

In [ ]:
# from google.colab import auth
# from google.cloud import bigquery
# import pandas as pd

# # This will prompt you to log in to your Google account
# auth.authenticate_user()

# # Initialize the BigQuery client
# project_id = 'mimic-iv-build' # Your project ID
# client = bigquery.Client(project=project_id)

## data loading, loading final_features from bigquery

In [ ]:
# query = """
# SELECT * FROM `mimic-iv-build.my_dataset.final_features`
# ORDER BY stay_id, hr
# """

# df = client.query(query).to_dataframe()

# # Show the first few rows
# df.head()

In [ ]:
# Save to the Colab virtual machine
#df.to_csv('my_sepsis_data.csv', index=False)

In [ ]:
#df.to_parquet('raw_data.parquet', index=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
#df = pd.read_parquet('raw_data.parquet')
df = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/bakatoo/raw_data.parquet')

In [ ]:
how_many_patients = df.groupby('stay_id')['hadm_id'].nunique()
how_many_patients = how_many_patients.sum()
sepsis = df.groupby('stay_id')['sepsis3'].max()
how_many_sepsis = sepsis.sum()
how_many_non_sepsis = how_many_patients - how_many_sepsis

print(f"Patients: {how_many_patients}")
print(f"Sepsis: {how_many_sepsis}")
print(f"Non-sepsis: {how_many_non_sepsis}")

Patients: 65351
Sepsis: 27881
Non-sepsis: 37470


## analysis icustay hourly timeline limitation check

In [ ]:
# query = """
# WITH icustay_times AS (
#   SELECT
#     stay_id,
#     MIN(charttime) AS first_hr_time,
#     MAX(charttime) AS last_hr_time
#   FROM `physionet-data.mimiciv_3_1_icu.chartevents`
#   WHERE itemid = 220045  -- heart rate
#   GROUP BY stay_id
# ),
# comparison AS (
#   SELECT
#     i.stay_id,
#     i.intime AS actual_intime,
#     i.outtime AS actual_outtime,
#     t.first_hr_time,
#     t.last_hr_time,
#     ROUND(DATETIME_DIFF(t.first_hr_time, i.intime, MINUTE) / 60.0, 2) AS hours_lost_start,
#     ROUND(DATETIME_DIFF(i.outtime, t.last_hr_time, MINUTE) / 60.0, 2) AS hours_lost_end
#   FROM `physionet-data.mimiciv_3_1_icu.icustays` i
#   LEFT JOIN icustay_times t ON i.stay_id = t.stay_id
# )
# SELECT
#   COUNT(*) AS total_patients,
#   ROUND(AVG(hours_lost_start), 2) AS avg_hours_lost_start,
#   ROUND(AVG(hours_lost_end), 2) AS avg_hours_lost_end,
#   ROUND(MAX(hours_lost_start), 2) AS max_hours_lost_start,
#   ROUND(MAX(hours_lost_end), 2) AS max_hours_lost_end,
#   COUNTIF(hours_lost_start > 2) AS patients_missing_2plus_hours_start,
#   COUNTIF(hours_lost_end > 2) AS patients_missing_2plus_hours_end,
#   COUNTIF(hours_lost_start > 6) AS patients_missing_6plus_hours_start,
#   COUNTIF(hours_lost_end > 6) AS patients_missing_6plus_hours_end
# FROM comparison
# """

# impact_df = client.query(query).to_dataframe()
# print(impact_df)

## sepsis-specific timeline analysis



In [ ]:
# query = """
# WITH icustay_times AS (
#   SELECT
#     stay_id,
#     MIN(charttime) AS first_hr_time
#   FROM `physionet-data.mimiciv_3_1_icu.chartevents`
#   WHERE itemid = 220045
#   GROUP BY stay_id
# ),
# comparison AS (
#   SELECT
#     i.stay_id,
#     ROUND(DATETIME_DIFF(t.first_hr_time, i.intime, MINUTE) / 60.0, 2) AS hours_lost_start
#   FROM `physionet-data.mimiciv_3_1_icu.icustays` i
#   LEFT JOIN icustay_times t ON i.stay_id = t.stay_id
# )
# SELECT
#   COUNT(*) AS sepsis_patients_affected,
#   ROUND(AVG(hours_lost_start), 2) AS avg_hours_lost,
#   COUNTIF(hours_lost_start > 2) AS missing_2plus_hours,
#   COUNTIF(hours_lost_start > 6) AS missing_6plus_hours
# FROM comparison
# WHERE stay_id IN (
#   SELECT DISTINCT stay_id
#   FROM `mimic-iv-build.my_dataset.final_features`
#   WHERE sepsis3 = 1
# )
# """
# sepsis_impact = client.query(query).to_dataframe()
# print(sepsis_impact)

# Preprocessing

## Gender -> M,F -> 1,0

In [ ]:
df['gender'] = df['gender'].map({'M': 1, 'F': 0})

## Preprocessing step 1 - onset_dt computation

For each sepsis patient, identifies the first hour where sepsis3 = 1, assigns it as onset_dt

In [ ]:
## Precondition Identifying first hour where sepsis3 == 1 per stay, only to sepsis patients
onset_df = (
    df[df['sepsis3'] == 1]
    .groupby('stay_id')['chart_hour']
    .min()
    .reset_index()
    .rename(columns={'chart_hour': 'onset_dt'})
)

# Merge back
df = df.merge(onset_df, on='stay_id', how='left')


In [ ]:
df.head()

,subject_id,stay_id,hadm_id,icu_intime,icu_outtime,age,gender,hr,chart_hour,lactate,...,gcs_unable,alt,ast,ptt,inr,po2,fio2,bicarbonate,sepsis3,onset_dt
0,12466550,30000153,23998182,2174-09-29 12:09:00,2174-10-01 03:26:10,61,1,0,2174-09-29 13:00:00,NaN,...,1,NaN,NaN,NaN,NaN,NaN,75.0,NaN,0,NaT
1,12466550,30000153,23998182,2174-09-29 12:09:00,2174-10-01 03:26:10,61,1,1,2174-09-29 14:00:00,1.3,...,<NA>,NaN,NaN,NaN,NaN,221.0,NaN,NaN,0,NaT
2,12466550,30000153,23998182,2174-09-29 12:09:00,2174-10-01 03:26:10,61,1,2,2174-09-29 15:00:00,2.1,...,<NA>,NaN,NaN,NaN,NaN,263.0,NaN,NaN,0,NaT
3,12466550,30000153,23998182,2174-09-29 12:09:00,2174-10-01 03:26:10,61,1,3,2174-09-29 16:00:00,NaN,...,<NA>,NaN,NaN,25.3,1.1,NaN,50.0,19.0,0,NaT
4,12466550,30000153,23998182,2174-09-29 12:09:00,2174-10-01 03:26:10,61,1,4,2174-09-29 17:00:00,NaN,...,1,NaN,NaN,NaN,NaN,215.0,NaN,NaN,0,NaT


## Preprocessing step 2 - remove rows after sepsis onset

In [ ]:
## 2 removing all rows after the sepsis onset turned to 1, keeping the onset hour

df = df[
    (df['onset_dt'].isna()) |
    (df['chart_hour'] <= df['onset_dt'])
]

## Preprocessing step 3 - feature range inspecton

In [ ]:
## Applying correct ranges for each feature, that are physically plausible

features = [
    "lactate","ph","heart_rate","temp_c","respiratory_rate",
    "sbp_cuff","dbp_cuff","map_cuff",
    "sbp_arterial","dbp_arterial","map_arterial",
    "wbc","neutrophil_pct","neutrophil_abs",
    "creatinine","bun","spo2","sao2",
    "albumin","bilirubin","crp",
    "sodium","potassium","chloride","calcium","magnesium",
    "glucose","gcs_total", "gcs_eyes", "gcs_verbal", "gcs_motor",
    "ptt", "inr",
    "alt","ast","po2","fio2","bicarbonate"
]

summary = df[features].describe(percentiles=[0.01,0.05,0.95,0.99]).T
summary

,count,mean,std,min,1%,5%,50%,95%,99%,max
lactate,50985.0,2.440188,2.342598,0.200000,0.6000,0.800000,1.700000,6.500000,13.300000,2.920000e+01
ph,86636.0,7.369326,0.082603,6.560000,7.1000,7.230000,7.380000,7.480000,7.530000,7.800000e+00
heart_rate,1877543.0,82.206159,21.954917,0.000000,47.5000,56.000000,80.400000,114.000000,130.500000,1.133700e+04
temp_c,567149.0,36.905392,8.724929,-17.777778,35.3750,36.055556,36.833333,37.777778,38.500000,5.445556e+03
respiratory_rate,1824405.0,19.222754,404.077865,0.000000,9.0000,12.000000,18.000000,28.000000,34.000000,5.456700e+05
sbp_cuff,1314040.0,123.088118,189.992709,-2.000000,82.0000,91.000000,121.000000,160.000000,178.000000,1.411460e+05
dbp_cuff,1313879.0,69.559346,257.479070,0.000000,37.0000,45.000000,67.000000,96.000000,112.000000,1.001070e+05
map_cuff,1316637.0,97.572121,11076.718731,0.000000,51.0000,59.000000,81.000000,110.000000,125.000000,8.999090e+06
sbp_arterial,455712.0,121.393998,21.756917,-94.000000,80.0000,91.000000,119.000000,159.000000,179.981667,3.175000e+03
dbp_arterial,455655.0,60.904895,206.121895,-40.000000,36.0000,42.000000,58.000000,83.000000,98.000000,7.410000e+04


## Preprocessing step 4 - fio2 unit harmonisation

In [ ]:
#STEP 0 - FiO2 unit harmonisation, this is based on derived
import numpy as np

if 'fio2' in df.columns:
    n_before = df['fio2'].notna().sum()

    fio2_harmonised = df['fio2'].copy()

    # fractional format → convert to percentage
    fio2_harmonised = np.where(
        (df['fio2'] > 0.2) & (df['fio2'] <= 1.0),
        df['fio2'] * 100,
        fio2_harmonised
    )

    df['fio2'] = fio2_harmonised

    n_after = df['fio2'].notna().sum()
    print(f"FiO2 harmonisation: {n_before} values before, "
          f"{n_after} after, {n_before - n_after} removed")

    # Inspect result
    print(df['fio2'].describe(percentiles=[0.005, 0.01, 0.99, 0.995]))

FiO2 harmonisation: 147445 values before, 147445 after, 0 removed
count    147445.000000
mean         52.018904
std          24.051268
min           0.000000
0.5%          4.000000
1%           22.000000
50%          50.000000
99%         100.000000
99.5%       100.000000
max        4050.000000
Name: fio2, dtype: float64


## Preprocessing step 5 - remove physiologically impossible values

In [ ]:
# STEP 1 - remove truly impossible values (before split applied)

impossible_bounds = {
    "lactate": (0, None),
    "ph": (0, 14),
    "heart_rate": (0, None),
    "temp_c": (0, None),
    "respiratory_rate": (0, None),
    "sbp_cuff": (0, None),
    "dbp_cuff": (0, None),
    "map_cuff": (0, None),
    "sbp_arterial": (0, None),
    "dbp_arterial": (0, None),
    "map_arterial": (0, None),
    "wbc": (0, None),
    "neutrophil_pct": (None, 100),
    "neutrophil_abs": (0, None),
    "creatinine": (0, None),
    "bun": (0, None),
    "spo2": (0, 100),
    "sao2": (0, 100),
    "albumin": (0, None),
    "bilirubin": (0, None),
    "crp": (0, None),
    "sodium": (0, None),
    "potassium": (0, None),
    "chloride": (0, None),
    "calcium": (0, None),
    "magnesium": (0, None),
    "glucose": (0, None),
    "gcs_total": (None, 15),
    "gcs_eyes": (None, 4),
    "gcs_verbal": (None, 5),
    "gcs_motor": (None, 6),
    "ptt": (0, None),
    "inr": (0, None),
    "alt": (0, None),
    "ast": (0, None),
    "po2": (0, None),
    "fio2": (20, 100),
    "bicarbonate": (0, None)
}

# Apply bounds and track removals
removal_log = {}

for col, (lo, hi) in impossible_bounds.items():
    if col not in df.columns:
        print(f"WARNING: {col} not found in dataframe, skipping")
        continue

    n_lo, n_hi = 0, 0

    if lo is not None:
        #find all rows where value too low
        mask_lo = df[col] <= lo
        #sums them together
        n_lo = mask_lo.sum()
        #replace them with NaN
        df.loc[mask_lo, col] = np.nan

    if hi is not None:
        mask_hi = df[col] > hi
        n_hi = mask_hi.sum()
        df.loc[mask_hi, col] = np.nan

    removal_log[col] = {
        'removed_below': n_lo,
        'removed_above': n_hi,
        'total_removed': n_lo + n_hi
    }

# Summary table
removal_summary = pd.DataFrame(removal_log).T
removal_summary = removal_summary[removal_summary['total_removed'] > 0]
print("=== Impossible value removal summary ===")
print(removal_summary.sort_values('total_removed', ascending=False))

=== Impossible value removal summary ===
                  removed_below  removed_above  total_removed
respiratory_rate           1992              0           1992
fio2                        976              6            982
heart_rate                  273              0            273
map_arterial                268              0            268
spo2                         76             53            129
temp_c                       58              0             58
sbp_cuff                     35              0             35
dbp_arterial                 27              0             27
sbp_arterial                 27              0             27
dbp_cuff                     26              0             26
neutrophil_abs               23              0             23
map_cuff                     19              0             19
creatinine                    7              0              7
bilirubin                     5              0              5
calcium                      

### inspection map_arterial for zero and negative values

In [ ]:
# Load a fresh copy just for inspection, don't overwrite df
print(df['map_arterial'].describe())
print("Exactly zero:", (df['map_arterial'] == 0).sum())
print("Negative:", (df['map_arterial'] < 0).sum())  # remove from memory after checking

count    458868.000000
mean         81.124423
std         179.737614
min           0.500000
25%          70.000000
50%          78.000000
75%          89.000000
max      115105.000000
Name: map_arterial, dtype: float64
Exactly zero: 0
Negative: 0


## Preprocessing step 6 - train/test split

Splits the dataset into training (80%) and test (20%) sets

In [ ]:
#STEP 2: train/test split by stay-id
# STEP 2 — train/test split by stay_id (80/20)
from sklearn.model_selection import train_test_split

stay_ids = df['stay_id'].unique()


train_ids, test_ids = train_test_split(
    stay_ids,
    test_size=0.2,
    random_state=42  # for reproducibility
)

train_df = df[df['stay_id'].isin(train_ids)].copy()
test_df  = df[df['stay_id'].isin(test_ids)].copy()

print(f"Total stays:      {len(stay_ids)}")
print(f"Train stays:      {len(train_ids)} ({len(train_ids)/len(stay_ids)*100:.1f}%)")
print(f"Test stays:       {len(test_ids)} ({len(test_ids)/len(stay_ids)*100:.1f}%)")
print(f"Train rows:       {len(train_df)}")
print(f"Test rows:        {len(test_df)}")

Total stays:      65351
Train stays:      52280 (80.0%)
Test stays:       13071 (20.0%)
Train rows:       1666377
Test rows:        413817


In [ ]:
# Quick sanity check - these should all be 0
print("Zeros in train map_arterial:",
      (train_df['map_arterial'] == 0).sum())
print("Negatives in train map_arterial:",
      (train_df['map_arterial'] < 0).sum())
print("Values > 100 in train spo2:",
      (train_df['spo2'] > 100).sum())

# And check the scary max value is still there
# (will be caught in next step by percentiles)
print("Max map_arterial in train:",
      train_df['map_arterial'].max())

Zeros in train map_arterial: 0
Negatives in train map_arterial: 0
Values > 100 in train spo2: 0
Max map_arterial in train: 115105.0


### validation check sepsis prevalence in train vs test

In [ ]:
# Check sepsis prevalence is similar in train and test
# If very different, your split may have introduced imbalance
train_sepsis = train_df.groupby('stay_id')['sepsis3'].max()
test_sepsis  = test_df.groupby('stay_id')['sepsis3'].max()

print(f"Train sepsis rate: "
      f"{train_sepsis.sum()} / {len(train_sepsis)} stays "
      f"({train_sepsis.mean()*100:.1f}%)")

print(f"Test sepsis rate:  "
      f"{test_sepsis.sum()} / {len(test_sepsis)} stays "
      f"({test_sepsis.mean()*100:.1f}%)")

Train sepsis rate: 22321 / 52280 stays (42.7%)
Test sepsis rate:  5560 / 13071 stays (42.5%)


## Preprocessing step 7 - percentile insepction on TRAIN only

In [ ]:
#STEP 3 - compute percentiles on TRAIN only
##

features = [
    "lactate","ph","heart_rate","temp_c","respiratory_rate",
    "sbp_cuff","dbp_cuff","map_cuff",
    "sbp_arterial","dbp_arterial","map_arterial",
    "wbc","neutrophil_pct","neutrophil_abs",
    "creatinine","bun","spo2","sao2",
    "albumin","bilirubin","crp",
    "sodium","potassium","chloride","calcium","magnesium",
    "glucose","gcs_total", "gcs_eyes", "gcs_verbal", "gcs_motor",
    "ptt", "inr",
    "alt","ast","po2","fio2","bicarbonate"
]

percentile_summary = train_df[features].describe(
    percentiles=[0.005, 0.01, 0.99, 0.995]
).T[['min', '0.5%', '1%', '99%', '99.5%', 'max']]

print("=== Percentile inspection on train set ===")
print(percentile_summary.to_string())

=== Percentile inspection on train set ===
                        min    0.5%          1%        99%        99.5%           max
lactate            0.200000    0.50    0.600000    13.6000    16.543000  2.920000e+01
ph                 6.560000    7.03    7.100000     7.5300     7.550000  7.800000e+00
heart_rate         1.000000   44.00   47.000000   131.0000   138.000000  1.133700e+04
temp_c             0.111111   35.00   35.388889    38.5000    38.888889  5.445556e+03
respiratory_rate   0.600000    8.00    9.000000    34.0000    37.000000  5.456700e+05
sbp_cuff           2.000000   78.00   82.000000   178.0000   185.000000  1.411460e+05
dbp_cuff           5.000000   34.00   37.000000   112.0000   119.000000  1.001070e+05
map_cuff           2.000000   48.00   51.000000   125.0000   132.000000  8.999090e+06
sbp_arterial       1.000000   74.00   80.000000   179.0000   187.000000  3.175000e+03
dbp_arterial       1.000000   33.00   36.000000    98.0000   105.000000  6.810500e+04
map_arteria

## Preprocessing step 8 final bounds removal

In [ ]:
# STEP 4 — apply final bounds to BOTH train and test
# Based on: Harutyunyan where available, 99.5th percentile where not

final_bounds = {
    "lactate":          (0, 33),
    "ph":               (6.3, 10),
    "heart_rate":       (0, 390),
    "temp_c":           (14.2, 47),
    "respiratory_rate": (0, 330),
    "sbp_cuff":         (0, 400),
    "dbp_cuff":         (0, 375),
    "map_cuff":         (0, 375),
    "sbp_arterial":     (0, 400),
    "dbp_arterial":     (0, 375),
    "map_arterial":     (0, 375),
    "wbc":              (0, 1100),
    "neutrophil_pct":   (0, 100),
    "neutrophil_abs":   (0, 45.34),   # 99.5th percentile
    "creatinine":       (0, 150),
    "bun":              (0, 300),
    "spo2":             (0, 100),
    "sao2":             (0, 100),
    "albumin":          (0, 60),
    "bilirubin":        (0, 66),
    "crp":              (0, 292.34),  # 99.5th percentile
    "sodium":           (0, 250),
    "potassium":        (0, 30),
    "chloride":         (0, 200),
    "calcium":          (0, 10.80),   # 99.5th percentile
    "magnesium":        (0, 22),
    "glucose":          (0, 2200),
    "gcs_total":        (None, 15),
    "gcs_eyes":         (None, 4),
    "gcs_verbal":       (None, 5),
    "gcs_motor":        (None, 6),
    "ptt":              (0, 150),
    "inr":              (0, 150),
    "alt":              (0, 11000),
    "ast":              (0, 22000),
    "po2":              (0, 770),
    "fio2":             (20, 100),
    "bicarbonate":      (0, 66),
}

# Apply to BOTH train and test using train-derived bounds
removal_log_final = {}

for col, (lo, hi) in final_bounds.items():
    n_lo_train, n_hi_train = 0, 0
    n_lo_test,  n_hi_test  = 0, 0

    if lo is not None:
        # train
        mask = train_df[col] <= lo
        n_lo_train = mask.sum()
        train_df.loc[mask, col] = np.nan
        # test
        mask = test_df[col] <= lo
        n_lo_test = mask.sum()
        test_df.loc[mask, col] = np.nan

    if hi is not None:
        # train
        mask = train_df[col] > hi
        n_hi_train = mask.sum()
        train_df.loc[mask, col] = np.nan
        # test
        mask = test_df[col] > hi
        n_hi_test = mask.sum()
        test_df.loc[mask, col] = np.nan

    removal_log_final[col] = {
        'train_removed': n_lo_train + n_hi_train,
        'test_removed':  n_lo_test  + n_hi_test,
    }

# Summary
removal_final_summary = pd.DataFrame(removal_log_final).T
removal_final_summary = removal_final_summary[
    removal_final_summary['train_removed'] +
    removal_final_summary['test_removed'] > 0
]
print("=== Final bounds removal summary ===")
print(removal_final_summary.sort_values('train_removed', ascending=False))

=== Final bounds removal summary ===
                  train_removed  test_removed
calcium                     493           119
temp_c                      220            41
dbp_cuff                     72            19
alt                          46             6
map_cuff                     45            12
neutrophil_pct               45            12
neutrophil_abs               38             8
dbp_arterial                 27             4
sbp_cuff                     12             4
magnesium                    11             3
map_arterial                 10             3
crp                           7             3
heart_rate                    6             2
ast                           6             2
respiratory_rate              5             4
sbp_arterial                  2             0


In [ ]:
print("chart_hour dtype:", train_df['chart_hour'].dtype)
print("onset_dt dtype:",   train_df['onset_dt'].dtype)
print("\nchart_hour sample:", train_df['chart_hour'].head(3).values)
print("onset_dt sample:",   train_df['onset_dt'].head(3).values)

chart_hour dtype: datetime64[us]
onset_dt dtype: datetime64[us]

chart_hour sample: ['2174-09-29T13:00:00.000000' '2174-09-29T14:00:00.000000'
 '2174-09-29T15:00:00.000000']
onset_dt sample: ['NaT' 'NaT' 'NaT']


## saving train and test to parquet

In [ ]:
# save preprocessed train_df and test_df for sliding window
# no need to rerun the whole notebook next time
train_df.to_parquet('/content/drive/MyDrive/Colab Notebooks/bakatoo/train_df_preprocessed.parquet', index=False)
test_df.to_parquet('/content/drive/MyDrive/Colab Notebooks/bakatoo/test_df_preprocessed.parquet', index=False)
print(f"Saved train_df: {train_df.shape}")
print(f"Saved test_df:  {test_df.shape}")

Saved train_df: (1666377, 51)
Saved test_df:  (413817, 51)
